# BirdCLEF+ 2026 — Stage 3.1: Mel-CNN student (GPU)

Trains a **MobileNetV3-Small mel-CNN** on **127k 5s windows** distilled from the v3
Perch+priors+MLP-probe+attn-temporal+student teacher (`full_cache_probs`), blended
with the 708 hard-labeled rows.

* **Why CNN on top of Perch?** The 708 labeled rows come from only 8 sites (dominated
  by S22). The 127k cache spans 23 sites including 15 with *no* labeled data. The v3
  by-site OOF was 0.8477 vs by-file 0.9495 (-0.099 drop). A mel-CNN trained on the
  full cross-site cache sees those sites directly, so it should narrow the PB gap.

* **Budget**: Kaggle T4 (~15 GB), ≤ 9 h session, 30 h/week GPU quota. One seed takes
  ~45-60 min at 30 epochs, so 3 seeds fit in a single session.

* **Output**: per-seed ONNX (fp16) ≤ 5 MB each, packed as a Kaggle dataset for v4.

### Required Kaggle inputs (add in the notebook sidebar)

1. `birdclef-2026` — competition audio (`train_soundscapes/*.ogg`)
2. `tiantanghuaxiao/birdclef-2026` — our v3 submission bundle (we reuse its
   `teacher_cache.pkl` equivalent payload for distillation targets)

### What the notebook assumes is uploaded in the v3 dataset root (or a sibling one)

* `teacher_cache_distill.pkl` — small wrapper with:
  * `meta_row_id` : list[str], length 127896
  * `full_cache_probs` : uint8 (quantized teacher probs) or float16, shape (127896, 234)
  * `primary_labels` : list[str], length 234
  * `labeled_cache_idx` : int32[708] positions of labeled rows in the cache
  * `Y_full_truth`     : uint8[708, 234]

We prepare that wrapper with `scripts/07_prepare_cnn_distill.py` *before* uploading
the dataset version used by this notebook.

## 1. Environment + deterministic seeds

In [ ]:
import os, sys, gc, time, math, json, random, pickle
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print('python :', sys.version.split()[0])
print('torch  :', torch.__version__, 'cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

assert torch.cuda.is_available(), 'This notebook requires a GPU session (T4 or better).'
DEVICE = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

## 2. Config + paths

Edit `BASE` to match the actual Kaggle mount points if you rename the dataset.

In [ ]:
# Kaggle paths — this workspace uses the `/kaggle/input/competitions/...` and
# `/kaggle/input/datasets/<user>/<slug>/...` layout (same as submit.ipynb). We
# probe both that and the flat `/kaggle/input/<slug>/` layout so the notebook
# keeps working if Kaggle flips the path convention.
_COMP_CANDIDATES = [
    Path('/kaggle/input/competitions/birdclef-2026'),
    Path('/kaggle/input/birdclef-2026'),
]
COMP = next((p for p in _COMP_CANDIDATES if p.exists()), _COMP_CANDIDATES[0])

_DISTILL_CANDIDATES = [
    Path('/kaggle/input/datasets/tiantanghuaxiao/birdclef-2026-distill/teacher_cache_distill.pkl'),
    Path('/kaggle/input/datasets/tiantanghuaxiao/birdclef-2026/teacher_cache_distill.pkl'),
    Path('/kaggle/input/birdclef-2026-distill/teacher_cache_distill.pkl'),
    Path('/kaggle/input/birdclef-2026-cnn/teacher_cache_distill.pkl'),
    Path('/kaggle/input/birdclef-2026/teacher_cache_distill.pkl'),
]
DISTILL = next((p for p in _DISTILL_CANDIDATES if p.exists()), _DISTILL_CANDIDATES[0])

TRAIN_SOUND = COMP / 'train_soundscapes'
print('COMP             :', COMP,         '->', COMP.exists())
print('DISTILL          :', DISTILL,      '->', DISTILL.exists())
print('train_soundscapes:', TRAIN_SOUND,  '->', TRAIN_SOUND.exists(),
      f'({len(list(TRAIN_SOUND.glob("*.ogg"))) if TRAIN_SOUND.exists() else 0} files)')

OUT = Path('/kaggle/working')
OUT.mkdir(exist_ok=True)

In [ ]:
@dataclass
class CFG:
    sample_rate: int = 32000
    window_sec : float = 5.0
    window_len : int   = 160000  # sample_rate * window_sec
    n_mels     : int   = 128
    n_fft      : int   = 1024
    hop_length : int   = 320
    f_min      : int   = 40
    f_max      : int   = 15000
    top_db     : float = 80.0
    # Training
    batch_size : int   = 96
    num_workers: int   = 8
    epochs     : int   = 18
    lr         : float = 3e-4
    weight_decay: float = 1e-4
    grad_clip  : float = 5.0
    mixup_alpha: float = 0.4
    mixup_prob : float = 0.5
    # SpecAugment
    freq_mask_w: int   = 16
    time_mask_w: int   = 40
    spec_aug_n : int   = 2
    # Loss
    focal_gamma: float = 2.0
    hard_weight: float = 3.0    # upweight 708 labeled rows
    soft_weight: float = 1.0
    # Sampling: oversample windows with teacher max-prob >= this
    positive_threshold: float = 0.20
    positive_weight   : float = 3.0
    # Seeds for the 3 models
    seeds: tuple = (20260101, 20260215, 20260322)
    # Distill cache quantization:
    soft_prob_dtype: str = 'float16'  # set 'uint8' if we quantized on disk

cfg = CFG()
print(asdict(cfg))

## 3. Load distill cache + split index

We keep **all 127k windows** in the training pool; a small **file-level 5% held-out**
provides a sanity metric (macro AUC against teacher soft probs on held-out files).

In [ ]:
print('Loading distill cache:', DISTILL)
with open(DISTILL, 'rb') as f:
    D = pickle.load(f)

meta_row_id: list[str] = list(D['meta_row_id'])
full_probs: np.ndarray = D['full_cache_probs']
primary_labels: list[str] = list(D['primary_labels'])
labeled_cache_idx: np.ndarray = np.asarray(D['labeled_cache_idx'], dtype=np.int64)
Y_full_truth: np.ndarray = np.asarray(D['Y_full_truth'], dtype=np.uint8)
N, K = full_probs.shape
print('  windows :', N)
print('  classes :', K)
print('  labeled :', int(labeled_cache_idx.size))
print('  dtype   :', full_probs.dtype)

# row_id format: BC2026_Train_0001_S08_20250606_030007_<end_sec>
# stem = row_id rsplit once on '_', digits at end.
def parse_row_id(rid: str) -> tuple[str, int]:
    stem, end = rid.rsplit('_', 1)
    return stem, int(end)

row_stems, row_ends = [], np.empty(N, dtype=np.int32)
for i, rid in enumerate(meta_row_id):
    s, e = parse_row_id(rid)
    row_stems.append(s)
    row_ends[i] = e
unique_stems = sorted(set(row_stems))
stem_to_idx = {s: i for i, s in enumerate(unique_stems)}
stem_indices = np.array([stem_to_idx[s] for s in row_stems], dtype=np.int32)
print('  unique files :', len(unique_stems))

# Held-out 5% of files (deterministic) for sanity AUC
rng = np.random.default_rng(20260101)
perm = rng.permutation(len(unique_stems))
n_val_files = max(16, len(unique_stems) // 20)
val_file_mask = np.zeros(len(unique_stems), dtype=bool)
val_file_mask[perm[:n_val_files]] = True
val_mask = val_file_mask[stem_indices]
train_mask = ~val_mask
print('  train wins   :', int(train_mask.sum()))
print('  val   wins   :', int(val_mask.sum()))

In [ ]:
# Precompute hard-label presence for the 708 labeled rows
is_labeled = np.zeros(N, dtype=bool)
is_labeled[labeled_cache_idx] = True
hard_label_matrix = np.zeros((N, K), dtype=np.uint8)
hard_label_matrix[labeled_cache_idx] = Y_full_truth

# Teacher soft probs as float16 on disk; we decode per-batch (see Dataset below)
if full_probs.dtype == np.uint8:
    print('Teacher probs are uint8-quantized (0-255). Dataset decodes to float32/255.')
else:
    print('Teacher probs are', full_probs.dtype, '; no decoding overhead.')

# Oversampling weights
# Use teacher max-prob as a proxy for "active" window.
if full_probs.dtype == np.uint8:
    max_prob = full_probs.max(axis=1).astype(np.float32) / 255.0
else:
    max_prob = full_probs.max(axis=1).astype(np.float32)
sample_weights = np.where(max_prob >= cfg.positive_threshold,
                          cfg.positive_weight, 1.0).astype(np.float32)
sample_weights[is_labeled] = max(sample_weights[is_labeled].max(), cfg.positive_weight)
print('  active wins (>=%.2f) : %d / %d (%.1f%%)' % (
    cfg.positive_threshold, int((max_prob >= cfg.positive_threshold).sum()), N,
    100.0 * (max_prob >= cfg.positive_threshold).mean()))

## 4. Audio loader + on-GPU mel front-end

* CPU workers load raw 5s waveforms (`soundfile`, precise offset seek).
* Mel + SpecAugment + per-sample normalization happen on the GPU (fused with model fwd).

This keeps CPU workers from bottlenecking and lets us crank batch size.

In [ ]:
import soundfile as sf

# filename lookup: stem -> Path
def _stem_to_path(stem: str) -> Path:
    return TRAIN_SOUND / f'{stem}.ogg'

# Sanity: make sure a few files exist
for s in unique_stems[:3]:
    p = _stem_to_path(s)
    assert p.exists(), f'missing file: {p}'
print('file lookup OK')

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, indices, *, cfg: CFG, train: bool):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.cfg = cfg
        self.train = train
        self.sr = cfg.sample_rate
        self.window_len = cfg.window_len

    def __len__(self): return len(self.indices)

    def _load_window(self, stem: str, end_sec: int) -> np.ndarray:
        path = _stem_to_path(stem)
        # end_sec is the *end* of the 5s window (5, 10, 15, ...)
        end_sample = int(end_sec * self.sr)
        start_sample = end_sample - self.window_len
        if start_sample < 0:
            start_sample = 0
            end_sample = self.window_len
        try:
            audio, sr = sf.read(str(path), start=start_sample, stop=end_sample,
                                dtype='float32', always_2d=False)
            if sr != self.sr:
                # Shouldn't happen for this comp; bail with zeros
                return np.zeros(self.window_len, dtype=np.float32)
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
            if audio.shape[0] < self.window_len:
                pad = np.zeros(self.window_len - audio.shape[0], dtype=np.float32)
                audio = np.concatenate([audio, pad])
            elif audio.shape[0] > self.window_len:
                audio = audio[:self.window_len]
            return audio
        except Exception:
            return np.zeros(self.window_len, dtype=np.float32)

    def _decode_soft(self, cache_idx: int) -> np.ndarray:
        p = full_probs[cache_idx]
        if p.dtype == np.uint8:
            return (p.astype(np.float32) / 255.0)
        return p.astype(np.float32)

    def __getitem__(self, i: int):
        idx = int(self.indices[i])
        stem = row_stems[idx]
        end_sec = int(row_ends[idx])
        wav = self._load_window(stem, end_sec)
        y_soft = self._decode_soft(idx)
        y_hard = hard_label_matrix[idx].astype(np.float32)
        is_lab = 1.0 if is_labeled[idx] else 0.0
        return {
            'wav'    : torch.from_numpy(wav),
            'y_soft' : torch.from_numpy(y_soft),
            'y_hard' : torch.from_numpy(y_hard),
            'is_lab' : torch.tensor(is_lab, dtype=torch.float32),
        }

# Weighted sampler on the train split
def make_train_loader(seed: int):
    g = torch.Generator(); g.manual_seed(seed)
    train_idx = np.where(train_mask)[0]
    sw = sample_weights[train_idx]
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=torch.tensor(sw, dtype=torch.double),
        num_samples=len(train_idx),
        replacement=True,
        generator=g,
    )
    ds = WindowDataset(train_idx, cfg=cfg, train=True)
    return DataLoader(
        ds, batch_size=cfg.batch_size, sampler=sampler,
        num_workers=cfg.num_workers, pin_memory=True,
        persistent_workers=(cfg.num_workers > 0), prefetch_factor=2,
        drop_last=True,
    )

def make_val_loader():
    val_idx = np.where(val_mask)[0]
    ds = WindowDataset(val_idx, cfg=cfg, train=False)
    return DataLoader(
        ds, batch_size=cfg.batch_size * 2, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True,
        persistent_workers=(cfg.num_workers > 0),
    )

## 5. Model (MobileNetV3-Small, 1-channel mel input)

Uses `torchvision.models.mobilenet_v3_small`. We swap the 3-ch stem to 1-ch by copying
the RGB mean of the weights. Classifier head → `K` logits.

In [ ]:
import torchaudio
from torchvision.models import mobilenet_v3_small

class MelFrontEnd(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=cfg.sample_rate,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            f_min=cfg.f_min, f_max=cfg.f_max,
            n_mels=cfg.n_mels, power=2.0,
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=cfg.top_db)
        # SpecAugment (training only; applied after normalization)
        self.freq_mask_w = cfg.freq_mask_w
        self.time_mask_w = cfg.time_mask_w
        self.spec_aug_n  = cfg.spec_aug_n

    def forward(self, wav: torch.Tensor, training: bool) -> torch.Tensor:
        # CRITICAL: torchaudio.MelSpectrogram / STFT are NOT autocast-safe and
        # produce NaN under fp16. Force the whole front-end to fp32.
        with torch.amp.autocast('cuda', enabled=False):
            wav_f32 = wav.float()
            x = self.mel(wav_f32)                   # (B, n_mels, T)
            # belt-and-suspenders clamp before log10 (handles all-zero windows)
            x = torch.clamp(x, min=1e-6)
            x = self.to_db(x)
            # per-sample normalization with std floor
            mu = x.mean(dim=(-2, -1), keepdim=True)
            sd = x.std(dim=(-2, -1), keepdim=True).clamp_min(1e-3)
            x = (x - mu) / sd
            x = x.unsqueeze(1)                      # (B, 1, n_mels, T)
        if training and self.freq_mask_w > 0 and self.time_mask_w > 0:
            for _ in range(self.spec_aug_n):
                f = int(torch.randint(0, self.freq_mask_w + 1, (1,)).item())
                f0 = int(torch.randint(0, max(1, x.shape[-2] - f), (1,)).item()) if f > 0 else 0
                if f > 0: x[..., f0:f0 + f, :] = 0
                t = int(torch.randint(0, self.time_mask_w + 1, (1,)).item())
                t0 = int(torch.randint(0, max(1, x.shape[-1] - t), (1,)).item()) if t > 0 else 0
                if t > 0: x[..., :, t0:t0 + t] = 0
        return x

class MelCNN(nn.Module):
    def __init__(self, n_classes: int, cfg: CFG):
        super().__init__()
        self.front = MelFrontEnd(cfg)
        backbone = mobilenet_v3_small(weights=None)
        # 1-channel stem: replace first conv
        old = backbone.features[0][0]
        new = nn.Conv2d(1, old.out_channels,
                        kernel_size=old.kernel_size,
                        stride=old.stride,
                        padding=old.padding,
                        bias=old.bias is not None)
        nn.init.kaiming_normal_(new.weight, mode='fan_out', nonlinearity='relu')
        backbone.features[0][0] = new
        # Replace classifier head to n_classes
        in_feat = backbone.classifier[0].in_features
        backbone.classifier = nn.Sequential(
            nn.Linear(in_feat, 1024),
            nn.Hardswish(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(1024, n_classes),
        )
        self.backbone = backbone

    def forward(self, wav: torch.Tensor, training: bool = False) -> torch.Tensor:
        x = self.front(wav, training=training)
        return self.backbone(x)

# sanity
m = MelCNN(K, cfg).cuda()
n_params = sum(p.numel() for p in m.parameters())
print(f'MelCNN params: {n_params/1e6:.2f}M')
with torch.no_grad():
    z = m(torch.randn(2, cfg.window_len, device='cuda'), training=False)
    print('logits:', z.shape)
del m; torch.cuda.empty_cache()

## 6. Loss (focal BCE, hard+soft blend) + mixup

In [ ]:
def focal_bce(logits, targets, gamma=2.0, reduction='mean', pos_weight=None):
    # binary focal loss on sigmoid(logits) vs targets in [0, 1]
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none',
                                             pos_weight=pos_weight)
    p = torch.sigmoid(logits)
    pt = targets * p + (1 - targets) * (1 - p)
    loss = ((1 - pt) ** gamma) * bce
    if reduction == 'mean': return loss.mean()
    if reduction == 'sum' : return loss.sum()
    return loss

def mixup_batch(wav, y_soft, y_hard, alpha):
    if alpha <= 0: return wav, y_soft, y_hard
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    idx = torch.randperm(wav.size(0), device=wav.device)
    wav_m    = lam * wav    + (1 - lam) * wav[idx]
    y_soft_m = lam * y_soft + (1 - lam) * y_soft[idx]
    y_hard_m = torch.clamp(lam * y_hard + (1 - lam) * y_hard[idx], 0.0, 1.0)
    return wav_m, y_soft_m, y_hard_m

## 7. Training loop (AMP + cosine LR)

In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def evaluate_val(model, loader):
    """Returns {'loss': float, 'auc': float, 'n_cols': int}.

    Primary metric is **val loss** (focal BCE vs teacher soft probs). Always
    finite — safe to use for model selection.

    AUC is a proxy: binarizes teacher probs at 0.3 and filters to classes
    that have both positives and negatives; can still be NaN when the val
    fold is too sparse, which is why we don't select on it.
    """
    import warnings
    from sklearn.metrics import roc_auc_score
    model.eval()
    ys, ps = [], []
    loss_sum, n_sum = 0.0, 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
        for batch in loader:
            wav = batch['wav'].cuda(non_blocking=True)
            y_soft = batch['y_soft'].cuda(non_blocking=True)
            logits = model(wav, training=False)
            l = focal_bce(logits, y_soft, gamma=cfg.focal_gamma, reduction='mean')
            loss_sum += float(l.item()) * wav.size(0)
            n_sum    += wav.size(0)
            ps.append(torch.sigmoid(logits).float().cpu().numpy())
            ys.append(batch['y_soft'].numpy())
    val_loss = loss_sum / max(1, n_sum)
    P = np.concatenate(ps, axis=0)
    T = np.concatenate(ys, axis=0)
    Tbin = (T >= 0.30).astype(np.int32)                       # lowered 0.5 -> 0.3
    pos  = Tbin.sum(axis=0)
    cols = np.where((pos > 0) & (pos < len(Tbin)))[0]          # need BOTH labels
    auc  = float('nan')
    if len(cols) > 0:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                auc = float(roc_auc_score(Tbin[:, cols], P[:, cols], average='macro'))
        except Exception:
            pass
    return {'loss': val_loss, 'auc': auc, 'n_cols': int(len(cols))}

def train_one_seed(seed: int, *, run_tag: str, epochs: int = None) -> dict:
    epochs = epochs or cfg.epochs
    set_seed(seed)
    train_loader = make_train_loader(seed=seed)
    val_loader   = make_val_loader()
    model = MelCNN(K, cfg).cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg.lr, total_steps=len(train_loader) * epochs,
        pct_start=0.05, anneal_strategy='cos', final_div_factor=25.0,
    )
    scaler = torch.amp.GradScaler('cuda')

    # Primary selection metric: val_loss (lower = better, always finite).
    # AUC kept as informational only (can be NaN on sparse val folds).
    best_val_loss = float('inf'); best_auc = float('nan'); best_epoch = -1; best_sd = None
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        running = 0.0; ntok = 0
        for it, batch in enumerate(train_loader):
            wav    = batch['wav'   ].cuda(non_blocking=True)
            y_soft = batch['y_soft'].cuda(non_blocking=True)
            y_hard = batch['y_hard'].cuda(non_blocking=True)
            is_lab = batch['is_lab'].cuda(non_blocking=True)
            if np.random.rand() < cfg.mixup_prob:
                wav, y_soft, y_hard = mixup_batch(wav, y_soft, y_hard, cfg.mixup_alpha)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                logits = model(wav, training=True)
                loss_soft = focal_bce(logits, y_soft, gamma=cfg.focal_gamma)
                # Hard loss only on labeled rows (mask via is_lab)
                loss_hard_per = focal_bce(logits, y_hard, gamma=cfg.focal_gamma,
                                          reduction='none').mean(dim=-1)
                loss_hard = (loss_hard_per * is_lab).sum() / (is_lab.sum() + 1e-6)
                loss = cfg.soft_weight * loss_soft + cfg.hard_weight * loss_hard
            if not torch.isfinite(loss):
                # Fail fast instead of burning GPU on NaN. If you see this in the
                # first few iters, the front-end / AMP interaction is the suspect.
                raise RuntimeError(
                    f'[{run_tag} s{seed} ep{epoch+1} it{it}] non-finite loss '
                    f'({loss.item()}); aborting this seed.')
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += float(loss.item()) * wav.size(0)
            ntok    += wav.size(0)
            if it % 200 == 0:
                mins = (time.time() - t0) / 60.0
                print(f'[{run_tag} s{seed} ep{epoch+1:02d} it{it:04d}] '
                      f'loss={running/max(1,ntok):.4f} lr={sched.get_last_lr()[0]:.2e} '
                      f'elapsed={mins:.1f}m')
        # Epoch-end eval (val loss is primary, AUC is informational)
        metric = evaluate_val(model, val_loader)
        mins = (time.time() - t0) / 60.0
        tag = '*' if metric['loss'] < best_val_loss else ' '
        print(f'[{run_tag} s{seed} ep{epoch+1:02d}] '
              f'val_loss={metric["loss"]:.4f}{tag} '
              f'val_auc={metric["auc"]:.4f} cols={metric["n_cols"]} '
              f'elapsed={mins:.1f}m')
        if metric['loss'] < best_val_loss:
            best_val_loss = metric['loss']
            best_auc      = metric['auc']
            best_epoch    = epoch + 1
            best_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Fallback: if for some reason every val_loss was non-finite, keep last weights.
    if best_sd is None:
        print(f'[{run_tag} s{seed}] WARN: no finite val_loss seen; saving final weights.')
        best_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epochs
    return {'best_val_loss': best_val_loss, 'best_auc': best_auc,
            'best_epoch': best_epoch, 'state_dict': best_sd, 'seed': seed,
            'minutes': (time.time() - t0) / 60.0}

## 8. Train 3 seeds

In [ ]:
runs = []
for i, seed in enumerate(cfg.seeds):
    print(f'\n============ SEED {i+1}/{len(cfg.seeds)} = {seed} ============')
    out = train_one_seed(seed, run_tag=f'm{i+1}')
    assert out['state_dict'] is not None, f'seed {seed}: empty state_dict, refusing to save'
    torch.save({'state_dict'  : out['state_dict'],
                'seed'         : seed,
                'best_val_loss': out['best_val_loss'],
                'best_auc'     : out['best_auc'],
                'best_epoch'   : out['best_epoch']},
               OUT / f'mel_cnn_seed{seed}.pt')
    runs.append({'seed': seed, 'best_val_loss': out['best_val_loss'],
                 'best_auc': out['best_auc'], 'best_epoch': out['best_epoch'],
                 'minutes': out['minutes']})
    print(f'  => saved mel_cnn_seed{seed}.pt  '
          f'(val_loss={out["best_val_loss"]:.4f}, auc={out["best_auc"]:.4f}, '
          f'ep={out["best_epoch"]}, {out["minutes"]:.1f}m)')
    torch.cuda.empty_cache(); gc.collect()

print('\nRUN SUMMARY:')
for r in runs: print(' ', r)

## 9. ONNX export (fp16, ≤ 5 MB each)

The exported ONNX includes the mel front-end so the inference notebook only needs to
feed raw 5s waveforms. We turn SpecAugment off by forcing `training=False`.

In [ ]:
class ExportWrapper(nn.Module):
    """Wraps MelCNN so that the forward takes just `wav` and always runs with
    `training=False`. Returns sigmoid probabilities so the deployed model is
    drop-in for fusion."""
    def __init__(self, base: MelCNN):
        super().__init__()
        self.base = base
    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        logits = self.base(wav, training=False)
        return torch.sigmoid(logits)

onnx_paths = []
for r in runs:
    seed = r['seed']
    model = MelCNN(K, cfg)
    sd = torch.load(OUT / f'mel_cnn_seed{seed}.pt', map_location='cpu')['state_dict']
    model.load_state_dict(sd)
    model.eval()
    wrap = ExportWrapper(model).eval()

    dummy = torch.randn(1, cfg.window_len)
    out_path = OUT / f'mel_cnn_seed{seed}.onnx'
    torch.onnx.export(
        wrap, dummy, out_path.as_posix(),
        input_names=['wav'], output_names=['prob'],
        dynamic_axes={'wav': {0: 'batch'}, 'prob': {0: 'batch'}},
        opset_version=17,
    )
    size_mb = out_path.stat().st_size / (1024 * 1024)
    print(f'  {out_path.name}  {size_mb:.2f} MB')
    onnx_paths.append(out_path)

# Quantize to fp16 (cuts size ~2x) using onnx converter
try:
    from onnxconverter_common import float16
    import onnx as _onnx
    for p in onnx_paths:
        m = _onnx.load(p.as_posix())
        m16 = float16.convert_float_to_float16(m, keep_io_types=True)
        q_path = p.with_name(p.stem + '_fp16.onnx')
        _onnx.save(m16, q_path.as_posix())
        print(f'  {q_path.name}  {q_path.stat().st_size / (1024*1024):.2f} MB')
except Exception as e:
    print('  (fp16 quantization skipped:', repr(e), ')')

## 10. Generate CNN-OOF on labeled rows (for fusion weight sweep)

We score the 708 labeled rows with each seed (ensemble mean) so the host pipeline
can sweep `alpha_cnn` in a fold-safe way. Because those 708 rows are in the *training*
set here (not held out), this is **in-sample** and will overestimate; the host pipeline
uses the exported ONNX to produce proper per-fold-safe predictions in a follow-up step.

For now we save the ensemble sigmoid on the 708 labeled rows and on the full 127k cache
so the host can inspect them.

In [ ]:
def score_indices(indices, seeds):
    """Ensemble sigmoid over seeds, returns (len(indices), K) float32."""
    ds = WindowDataset(indices, cfg=cfg, train=False)
    loader = DataLoader(ds, batch_size=cfg.batch_size * 2, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True)
    P_sum = np.zeros((len(indices), K), dtype=np.float32)
    for seed in seeds:
        model = MelCNN(K, cfg).cuda()
        sd = torch.load(OUT / f'mel_cnn_seed{seed}.pt', map_location='cuda')['state_dict']
        model.load_state_dict(sd); model.eval()
        off = 0
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
            for batch in loader:
                wav = batch['wav'].cuda(non_blocking=True)
                p = torch.sigmoid(model(wav, training=False)).float().cpu().numpy()
                P_sum[off:off+p.shape[0]] += p
                off += p.shape[0]
        del model; torch.cuda.empty_cache()
    return P_sum / len(seeds)

labeled_idx = np.where(is_labeled)[0]
P_labeled = score_indices(labeled_idx, cfg.seeds)
np.save(OUT / 'cnn_probs_labeled.npy', P_labeled)
print('saved cnn_probs_labeled.npy  shape', P_labeled.shape)

# Also write a small manifest
manifest = {
    'seeds': list(cfg.seeds),
    'runs' : runs,
    'n_classes': int(K),
    'sample_rate': int(cfg.sample_rate),
    'window_sec' : float(cfg.window_sec),
    'window_len' : int(cfg.window_len),
    'onnx'    : [p.name for p in onnx_paths],
    'primary_labels': primary_labels,
}
with open(OUT / 'cnn_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print('saved cnn_manifest.json')

## 11. (Optional) bundle prep: copy artifacts into a shareable folder

Run this cell only when you're happy with the results — it prints the files you should
add to the Kaggle dataset version for the inference notebook.

In [ ]:
keep = sorted(OUT.glob('mel_cnn_seed*_fp16.onnx')) or sorted(OUT.glob('mel_cnn_seed*.onnx'))
keep += [OUT / 'cnn_manifest.json', OUT / 'cnn_probs_labeled.npy']
keep = [p for p in keep if p.exists()]
total = sum(p.stat().st_size for p in keep) / (1024 * 1024)
print(f'\n  {len(keep)} files, total {total:.2f} MB:')
for p in keep:
    print(f'    {p.stat().st_size/(1024*1024):6.2f} MB  {p.name}')
print('\nUpload these into a new version of your Kaggle dataset, then point the')
print('submit notebook to load the ONNXs alongside submission_bundle.pkl (v3).')